# Job Market Intelligence Dashboard — EDA

## 1. Introduction

Analysis of **473 Data Analyst job postings** scraped from
[Himalayas.app](https://himalayas.app) (a remote-first job board with
permissive robots.txt and job-specific sitemaps), covering postings dated
**April 10 – July 7, 2026**.

Pipeline: `scripts/scraper.py` (schema.org JobPosting extraction, resumable) →
`scripts/clean.py` (dedup, currency/period-normalized salary, long-format
country/skill tables) → `scripts/load_database.py` (normalized SQLite,
`database/job_market.db`) → this notebook.

**Data source caveat:** Himalayas is remote-first — 447 of 473 postings are
Remote, 26 Hybrid, **0 On-site**. City-level location is essentially never
populated (remote roles don't have one), so geographic analysis here is at
the country level.

## 2. Data Loading

In [1]:
%matplotlib inline
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DB_PATH = Path("../database/job_market.db")
conn = sqlite3.connect(DB_PATH)

jobs = pd.read_sql("SELECT * FROM jobs", conn, parse_dates=["posting_date", "valid_through", "scraped_at"])
companies = pd.read_sql("SELECT * FROM companies", conn)
skills = pd.read_sql("SELECT * FROM skills", conn)
locations = pd.read_sql("SELECT * FROM locations", conn)
job_skills = pd.read_sql("""
    SELECT js.job_id, s.skill_name FROM job_skills js
    JOIN skills s ON js.skill_id = s.skill_id
""", conn)
job_locations = pd.read_sql("""
    SELECT jl.job_id, l.country_name FROM job_locations jl
    JOIN locations l ON jl.location_id = l.location_id
""", conn)
jobs = jobs.merge(companies, on="company_id", how="left")

for name, df in [("jobs", jobs), ("companies", companies), ("skills", skills),
                  ("locations", locations), ("job_skills", job_skills), ("job_locations", job_locations)]:
    print(f"{name:16s} {df.shape}")

import plotly.io as pio
pio.renderers.default = "notebook_connected"


jobs             (473, 22)
companies        (352, 3)
skills           (18, 2)
locations        (150, 2)
job_skills       (1663, 2)
job_locations    (818, 2)


## 3. Top Hiring Companies

In [2]:
top_companies = jobs["company_name"].value_counts().head(15).reset_index()
top_companies.columns = ["company", "postings"]

fig = px.bar(top_companies, x="postings", y="company", orientation="h",
             title="Top 15 Hiring Companies", color="postings", color_continuous_scale="Blues")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.show()
top_companies

,company,postings
0,TELUS Digital,9
1,Tabby,7
2,Capgemini Technology Services,6
3,YipitData,5
4,Paired,5
5,Lennor Group,5
6,The Kraft Heinz Company,4
7,Mercor,4
8,Pavago,4
9,CI&T,4


## 4. Most Requested Skills

In [3]:
skill_counts = job_skills["skill_name"].value_counts().reset_index()
skill_counts.columns = ["skill", "postings"]
skill_counts["pct_of_postings"] = (skill_counts["postings"] / len(jobs) * 100).round(1)

fig = px.bar(skill_counts, x="postings", y="skill", orientation="h",
             title="Most Requested Skills (share of 473 postings)",
             color="postings", color_continuous_scale="Teal")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=550)
fig.show()
skill_counts

,skill,postings,pct_of_postings
0,SQL,373,78.9
1,Python,223,47.1
2,Tableau,197,41.6
3,Power BI,186,39.3
4,Excel,184,38.9
5,Statistics,108,22.8
6,R,97,20.5
7,Snowflake,65,13.7
8,ETL,59,12.5
9,AWS,31,6.6


## 5. Most Common Job Titles

In [4]:
top_titles = jobs["job_title"].value_counts().head(15).reset_index()
top_titles.columns = ["job_title", "postings"]

fig = px.bar(top_titles, x="postings", y="job_title", orientation="h",
             title="Top 15 Job Titles", color="postings", color_continuous_scale="Purples")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.show()

## 6. Geographic Demand

City-level data isn't usable here (0% coverage — Himalayas postings are
remote-first and don't specify a city). Country-level demand instead, counted
via the `job_locations` junction so a posting open to 5 countries counts
toward all 5, not just one.

In [5]:
country_counts = job_locations["country_name"].value_counts().reset_index()
country_counts.columns = ["country", "postings"]

fig = px.choropleth(country_counts, locations="country", locationmode="country names", color="postings",
                     title="Data Analyst Demand by Country", color_continuous_scale="Blues")
fig.show()

fig2 = px.bar(country_counts.head(15), x="postings", y="country", orientation="h",
              title="Top 15 Countries by Posting Count", color="postings", color_continuous_scale="Blues")
fig2.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig2.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_10524\2443302390.py:4: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(country_counts, locations="country", locationmode="country names", color="postings",


## 7. Experience Level Distribution

In [6]:
exp_order = ["Entry Level", "Mid Level", "Senior Level", "Lead/Principal", "Not Specified"]
exp_counts = jobs["experience_level"].value_counts().reindex(exp_order).reset_index()
exp_counts.columns = ["experience_level", "postings"]

fig = px.pie(exp_counts, names="experience_level", values="postings", hole=0.45,
             title="Experience Level Distribution",
             category_orders={"experience_level": exp_order})
fig.show()

## 8. Remote vs Hybrid vs On-site

In [7]:
setup_counts = jobs["work_setup"].value_counts().reset_index()
setup_counts.columns = ["work_setup", "postings"]

fig = px.pie(setup_counts, names="work_setup", values="postings", hole=0.45,
             title="Work Setup Distribution (0 On-site in this remote-first dataset)")
fig.show()

## 9. Salary Analysis

Salaries are normalized to **annualized USD** during cleaning (period
annualized, currency converted via live exchange rates) so figures across
different countries/currencies/pay periods are directly comparable.

In [8]:
with_salary = jobs[jobs["salary_avg_usd_annual"].notna()]
print(f"{len(with_salary)} of {len(jobs)} postings ({len(with_salary)/len(jobs)*100:.0f}%) have usable salary data")

fig = px.histogram(with_salary, x="salary_avg_usd_annual", nbins=30,
                    title="Salary Distribution (Annualized USD)")
fig.update_layout(xaxis_title="Annual Salary (USD)", yaxis_title="Postings")
fig.show()

print(with_salary["salary_avg_usd_annual"].describe().round(0))

143 of 473 postings (30%) have usable salary data


count       143.0
mean     108283.0
std       80931.0
min        2511.0
25%       73265.0
50%       97300.0
75%      121750.0
max      650000.0
Name: salary_avg_usd_annual, dtype: float64


In [9]:
salary_by_skill = (jobs.merge(job_skills, on="job_id")
                   .dropna(subset=["salary_avg_usd_annual"])
                   .groupby("skill_name")["salary_avg_usd_annual"]
                   .agg(["mean", "count"]).round(0)
                   .query("count >= 3")
                   .sort_values("mean", ascending=False).reset_index())
salary_by_skill.columns = ["skill", "avg_salary_usd", "n_postings"]

fig = px.bar(salary_by_skill, x="avg_salary_usd", y="skill", orientation="h",
             title="Average Annual Salary by Skill (USD, min 3 postings)",
             color="avg_salary_usd", color_continuous_scale="Greens")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.show()
salary_by_skill

,skill,avg_salary_usd,n_postings
0,R,125367.0,46
1,Spark,118625.0,4
2,Python,118010.0,69
3,SQL,109586.0,113
4,Pandas,105958.0,5
5,Snowflake,102858.0,24
6,ETL,100378.0,9
7,Tableau,97331.0,62
8,Git,97241.0,8
9,Machine Learning,96663.0,6


In [10]:
salary_by_company = (with_salary.groupby("company_name")["salary_avg_usd_annual"]
                     .agg(["mean", "count"]).round(0)
                     .query("count >= 2")
                     .sort_values("mean", ascending=False).head(15).reset_index())
salary_by_company.columns = ["company", "avg_salary_usd", "n_postings"]

fig = px.bar(salary_by_company, x="avg_salary_usd", y="company", orientation="h",
             title="Average Salary by Company (USD, min 2 postings)",
             color="avg_salary_usd", color_continuous_scale="Oranges")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.show()

## 10. Salary vs Experience Level

In [11]:
fig = px.box(with_salary, x="experience_level", y="salary_avg_usd_annual",
             category_orders={"experience_level": exp_order},
             title="Salary Distribution by Experience Level")
fig.update_layout(xaxis_title="Experience Level", yaxis_title="Annual Salary (USD)")
fig.show()

# ordinal encoding for a numeric correlation coefficient
exp_rank = {"Entry Level": 1, "Mid Level": 2, "Senior Level": 3, "Lead/Principal": 4}
corr_df = with_salary[with_salary["experience_level"].isin(exp_rank)].copy()
corr_df["experience_rank"] = corr_df["experience_level"].map(exp_rank)
r = corr_df["experience_rank"].corr(corr_df["salary_avg_usd_annual"])
print(f"Pearson r (experience rank vs salary): {r:.3f}")

Pearson r (experience rank vs salary): 0.387


## 11. Top Skill Combinations

In [12]:
from itertools import combinations
from collections import Counter

pair_counter = Counter()
for _, group in job_skills.groupby("job_id")["skill_name"]:
    for pair in combinations(sorted(set(group)), 2):
        pair_counter[pair] += 1

top_pairs = pd.DataFrame(
    [(a, b, n) for (a, b), n in pair_counter.most_common(15)],
    columns=["skill_a", "skill_b", "co_occurrences"]
)
top_pairs["combo"] = top_pairs["skill_a"] + " + " + top_pairs["skill_b"]

fig = px.bar(top_pairs, x="co_occurrences", y="combo", orientation="h",
             title="Top 15 Skill Combinations", color="co_occurrences",
             color_continuous_scale="Magenta")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=550)
fig.show()

## 12. Monthly Hiring Trend

In [13]:
jobs["posting_month"] = jobs["posting_date"].dt.to_period("M").astype(str)
monthly = jobs.groupby("posting_month").size().reset_index(name="postings")

fig = px.line(monthly, x="posting_month", y="postings", markers=True,
              title="Postings by Month")
fig.update_layout(xaxis_title="Month", yaxis_title="Postings")
fig.show()
monthly

,posting_month,postings
0,2026-04,39
1,2026-05,154
2,2026-06,161
3,2026-07,119


## 13. Key Business Insights

- **SQL, Python, and Tableau dominate the market** — SQL appears in 79% of
  postings (373/473), Python in 47%, Tableau in 42%. A Data Analyst without
  SQL is targeting a small minority of roles.
- **Highest-paying skills skew technical/analytical**: R ($125K avg),
  Spark ($119K), Python ($118K) command a premium over Excel/Power BI-only
  roles — consistent with more technical roles paying more.
- **This job market is remote-first**: 0 on-site postings found at all;
  94% Remote, 6% Hybrid. Applicants shouldn't expect in-office Data Analyst
  roles from this particular source.
- **The United States dominates demand** (240 of 818 country-postings,
  ~29%), followed by the UK, India, Canada, and the Philippines — but nearly
  10% of postings (40/473) are open to multiple countries simultaneously,
  suggesting many employers aren't location-restrictive for this role.
- **Salary rises with seniority but the sample is salary-sparse**: only
  30% of postings disclose salary, so company- and skill-level salary
  comparisons should be read as directional, not definitive.
- **SQL + Python is the most common skill pairing**, followed by
  SQL + Tableau and SQL + Excel — reinforcing SQL as the near-universal
  baseline skill that pairs with a visualization or scripting specialty.

**Recommendation for aspiring Data Analysts**: prioritize SQL fluency first
(near-universal requirement), then specialize in either Python (higher
average pay, more technical roles) or Tableau/Power BI (broader
applicability across business-facing roles).

## 14. Next Steps

- **Power BI report**: build the executive-facing report (Stage 6) on top of
  `database/job_market.db` — same tables, richer interactive filtering.
- **Scheduler**: `scripts/scraper.py` is resumable by design (tracks
  `seen_urls.json`) — wire it into GitHub Actions on a weekly cron so the
  monthly hiring trend chart keeps accumulating real signal over time.
- **Documentation**: architecture diagram, data dictionary, and full README
  (Stage 7) once the Power BI layer is done.
- Consider expanding skill keyword coverage (e.g. dbt, Looker, Airflow) as
  new tools show up in postings.